In [54]:
import pandas as pd
import numpy as np


In [55]:
df_raw = pd.read_csv("data_aktual.csv")

In [56]:
df_raw.info()
df_raw.dtypes


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   tanggal                     7 non-null      object 
 1   Beras Kualitas Bawah I      5 non-null      float64
 2   Beras Kualitas Bawah II     5 non-null      float64
 3   Beras Kualitas Medium I     5 non-null      float64
 4   Beras Kualitas Medium II    5 non-null      float64
 5   Beras Kualitas Super I      5 non-null      float64
 6   Beras Kualitas Super II     5 non-null      float64
 7   Bawang Merah Ukuran Sedang  5 non-null      float64
 8   Cabai Rawit Hijau           5 non-null      float64
 9   Cabai Rawit Merah           5 non-null      float64
dtypes: float64(9), object(1)
memory usage: 692.0+ bytes


tanggal                        object
Beras Kualitas Bawah I        float64
Beras Kualitas Bawah II       float64
Beras Kualitas Medium I       float64
Beras Kualitas Medium II      float64
Beras Kualitas Super I        float64
Beras Kualitas Super II       float64
Bawang Merah Ukuran Sedang    float64
Cabai Rawit Hijau             float64
Cabai Rawit Merah             float64
dtype: object

In [57]:
df_cleaned = df_raw.copy()
kolom_numerik = df_cleaned.select_dtypes(include="number").columns
df_cleaned[kolom_numerik] = df_cleaned[kolom_numerik].interpolate(
    method="linear", limit_direction="forward", axis=0
)

In [58]:
df_cleaned.info()
df_cleaned.dtypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   tanggal                     7 non-null      object 
 1   Beras Kualitas Bawah I      7 non-null      float64
 2   Beras Kualitas Bawah II     7 non-null      float64
 3   Beras Kualitas Medium I     7 non-null      float64
 4   Beras Kualitas Medium II    7 non-null      float64
 5   Beras Kualitas Super I      7 non-null      float64
 6   Beras Kualitas Super II     7 non-null      float64
 7   Bawang Merah Ukuran Sedang  7 non-null      float64
 8   Cabai Rawit Hijau           7 non-null      float64
 9   Cabai Rawit Merah           7 non-null      float64
dtypes: float64(9), object(1)
memory usage: 692.0+ bytes


tanggal                        object
Beras Kualitas Bawah I        float64
Beras Kualitas Bawah II       float64
Beras Kualitas Medium I       float64
Beras Kualitas Medium II      float64
Beras Kualitas Super I        float64
Beras Kualitas Super II       float64
Bawang Merah Ukuran Sedang    float64
Cabai Rawit Hijau             float64
Cabai Rawit Merah             float64
dtype: object

In [59]:
from pathlib import Path

csv_path = Path("data_aktual_cleaned.csv")
df_baru = df_cleaned.copy()
if "tanggal" not in df_baru.columns:
    raise ValueError(
        "Kolom 'tanggal' wajib ada agar ekspor bisa digabung berdasarkan tanggal."
    )

df_baru["tanggal"] = pd.to_datetime(df_baru["tanggal"], errors="raise").dt.strftime(
    "%Y-%m-%d"
)
if df_baru["tanggal"].duplicated().any():
    raise ValueError("Tanggal pada df_cleaned harus unik.")

if csv_path.exists():
    df_lama = pd.read_csv(csv_path)
    if list(df_lama.columns) != list(df_baru.columns):
        raise ValueError("Kolom CSV lama berbeda dari df_cleaned; ekspor dibatalkan.")
    df_lama["tanggal"] = pd.to_datetime(df_lama["tanggal"], errors="raise").dt.strftime(
        "%Y-%m-%d"
    )
    df_lama = df_lama.drop_duplicates(subset="tanggal", keep="last")
    data_lama = df_lama.set_index("tanggal")
    data_baru = df_baru.set_index("tanggal")
    tanggal_baru = data_baru.index.difference(data_lama.index)
    tanggal_lama_baru = data_baru.index.intersection(data_lama.index)
    if len(tanggal_lama_baru):
        beda = data_baru.loc[tanggal_lama_baru].ne(data_lama.loc[tanggal_lama_baru])
        kedua_kosong = (
            data_baru.loc[tanggal_lama_baru].isna()
            & data_lama.loc[tanggal_lama_baru].isna()
        )
        berubah = (beda & ~kedua_kosong).any(axis=1)
        tanggal_berubah = berubah.index[berubah]
    else:
        tanggal_berubah = pd.Index([])

    tanggal_upsert = tanggal_baru.union(tanggal_berubah)
    if len(tanggal_upsert):
        df_export = pd.concat(
            [
                df_lama.loc[~df_lama["tanggal"].isin(tanggal_upsert)],
                data_baru.loc[tanggal_upsert].reset_index(),
            ],
            ignore_index=True,
        ).sort_values("tanggal", kind="stable")
        df_export.to_csv(csv_path, index=False)
        export_csv = (
            f"CSV diperbarui: {len(tanggal_baru)} tanggal baru, "
            f"{len(tanggal_berubah)} tanggal berubah."
        )
    else:
        export_csv = "Tidak ada perubahan; CSV tidak ditulis ulang."
else:
    df_export = df_baru.sort_values("tanggal")
    df_export.to_csv(csv_path, index=False)
    export_csv = f"CSV dibuat dengan {len(df_export)} tanggal."

export_csv

'CSV dibuat dengan 7 tanggal.'